# NBA Draft Outcome Predictor Modeling Notebook

**Goal**: Classify draft picks into: bust, rotation, starter, and star  
**Model**: XGBoost multiclass classifier  
**Data**: `nba.player_features` : one row per drafted player (2000–2018)

### Sections
1. Load & inspect
2. Preprocessing (imputation, encoding, train/test split)
3. Baseline model
4. Hyperparameter tuning
5. Evaluation (confusion matrix, per-class metrics)
6. Feature importance
7. Backtesting on known draft classes
8. Predict the 2025 draft class

In [ ]:
# !pip install xgboost scikit-learn pandas sqlalchemy psycopg2-binary matplotlib seaborn shap

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import shap

from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.3f}'.format)

RANDOM_STATE = 42
DB_URL = 'postgresql://localhost/nba_draft'   # ← update if needed

---
## 1. Load & Inspect

In [ ]:
engine = create_engine(DB_URL)

df = pd.read_sql("""
    SELECT
        pf.*,
        p.full_name,
        p.position
    FROM nba.player_features pf
    JOIN nba.players p USING (player_id)
    ORDER BY pf.draft_year, pf.pick_overall
""", engine)

print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Outcome distribution:  Checking for class distribution
outcome_counts = df['outcome'].value_counts()
print(outcome_counts)
print()
print(outcome_counts / len(df) * 100)

In [ ]:
# Null rates per column
# Combine cols (~40-50% null) are expected — players who skipped the combine
# College cols should be low null — flag anything >10%
null_rates = df.isnull().mean().sort_values(ascending=False)
null_rates[null_rates > 0].plot(
    kind='barh', figsize=(8, 6), title='Null Rate by Feature'
)
plt.axvline(0.1, color='red', linestyle='--', label='10% threshold')
plt.xlabel('Null rate')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature distributions by outcome — sanity check
# Stars should cluster at low pick numbers and high BPM
key_features = ['pick_overall', 'age_at_draft', 'adj_pts_per36', 'bpm', 'ts_pct', 'win_shares']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
outcome_order = ['star', 'starter', 'rotation', 'bust']
palette = {'star': '#FFD700', 'starter': '#4CAF50', 'rotation': '#2196F3', 'bust': '#F44336'}

for ax, feat in zip(axes.flat, key_features):
    sns.boxplot(
        data=df, x='outcome', y=feat,
        order=outcome_order, palette=palette, ax=ax
    )
    ax.set_title(feat)
    ax.set_xlabel('')

plt.suptitle('Feature Distributions by Outcome', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()